# Multi-Segment Streams

**Learning outcome:** Apply multi-segment streams through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Intermediate  
**Execution profile:** `base`  
**Expected runtime:** under 1 minute  
**Optional extras:** `notebook`

## Study question and data

**Study question:** How should a stream with changing heat-capacity flow be represented without hiding its temperature intervals?

## Step 1: Define and target a segmented stream

This cell defines an inline problem whose hot stream has two segments with different duties over adjacent temperature ranges. No utilities are supplied. It validates the input and runs `direct_heat_integration()`. The `assert` shows that `problem.master_zone` returns a detached copy, so changes must go through problem methods.

In [ ]:
from OpenPinch import PinchProblem

segmented_input = {
    "streams": [
        {"zone": "Site", "name": "Variable CP hot", "segments": [
            {"name": "hot-1", "t_supply": 200.0, "t_target": 150.0, "heat_flow": 50.0},
            {"name": "hot-2", "t_supply": 150.0, "t_target": 100.0, "heat_flow": 100.0},
        ]},
        {
            "zone": "Site",
            "name": "Cold demand",
            "t_supply": 40.0,
            "t_target": 170.0,
            "heat_flow": 130.0,
        },
    ],
    "utilities": [],
}
problem = PinchProblem(segmented_input, project_name="Site")
validated = problem.validate()
target = problem.target.direct_heat_integration()
# Observations are detached; use problem methods for changes.
site_zone = problem.master_zone
assert site_zone is not problem.master_zone
problem.summary_frame()

## Step 2: Audit the prepared segments

This cell reads back the prepared hot and cold streams and the utility lists. `segment_table` lists each hot segment's name, supply and target temperatures and duty. The `assert` confirms that each segment starts where the previous one ends. `composite` plots the composite curve so the segmented stream can be checked visually.

In [ ]:
prepared_hot = list(problem.hot_streams)[0]
prepared_cold = list(problem.cold_streams)[0]
hot_utilities = list(problem.hot_utilities)
cold_utilities = list(problem.cold_utilities)
segment_table = [
    {
        "name": segment.name,
        "supply": float(segment.supply_temperature),
        "target": float(segment.target_temperature),
        "duty": float(segment.heat_flow),
    }
    for segment in prepared_hot.segments
]
assert all(
    first.target_temperature == second.supply_temperature
    for first, second in zip(
        prepared_hot.segments, prepared_hot.segments[1:]
    )
)
composite = problem.plot.composite_curve()
segment_table

## Review the result

Inspect the prepared segment boundaries and duties beside the target summary before replacing the example with plant-specific variable-heat-capacity data.

In [ ]:
from IPython.display import display

display(segment_table)
display(problem.summary_frame())
display(composite)

## Interpret the result

Confirm adjacent segments are continuous and inspect each segment's temperature range and duty before trusting the target.

## Adapt this template

Add one segment per thermodynamic interval; do not average heat-capacity flow across phase or property changes.